# Video Recommendation Pipeline

This notebook trains a lightweight retriever and then feeds its sampled top-k candidates into a deep reranker using the same synthetic video catalog.

In [1]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
MODULE_DIR = ROOT 


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


lw = load_module("lightweight", MODULE_DIR / "lightweight.py")
dr = load_module("deepranker", MODULE_DIR / "deepranker.py")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
lw_cfg = lw.Config(
    num_users=1200,
    num_videos=400,
    num_topics=16,
    num_creators=80,
    train_size=1500,
    test_size=300,
    batch_size=96,
    epochs=2,
    retrieval_k=6,
    amp=False,
    compile_model=False,
    seed=7,
)

dr_cfg = dr.Config(
    num_users=lw_cfg.num_users,
    num_videos=lw_cfg.num_videos,
    num_topics=lw_cfg.num_topics,
    num_creators=lw_cfg.num_creators,
    user_dense_dim=lw_cfg.user_dense_dim,
    video_dense_dim=lw_cfg.video_dense_dim,
    slate_size=lw_cfg.retrieval_k,
    batch_size=lw_cfg.batch_size,
    epochs=3,
    amp=False,
    compile_model=False,
    seed=11,
)

lw.seed_everything(lw_cfg.seed)
dr.seed_everything(dr_cfg.seed)

lw_cfg, dr_cfg

(Config(num_users=1200, num_videos=400, user_dense_dim=10, video_dense_dim=12, num_topics=16, num_creators=80, train_size=1500, test_size=300, batch_size=96, epochs=2, lr=0.002, weight_decay=0.001, grad_clip=1.0, temperature=0.08, user_id_embed_dim=32, video_id_embed_dim=32, hidden_dim=96, embedding_dim=64, dropout=0.1, num_workers=0, amp=False, compile_model=False, retrieval_k=6, seed=7),
 Config(num_users=1200, num_videos=400, num_topics=16, num_creators=80, history_min_len=12, history_max_len=40, user_dense_dim=10, video_dense_dim=12, train_size=3500, test_size=800, slate_size=6, batch_size=96, epochs=3, lr=0.001, weight_decay=0.01, grad_clip=1.0, alpha=1.0, beta=0.25, video_embed_dim=48, topic_embed_dim=12, creator_embed_dim=12, hidden_dim=96, num_heads=4, num_layers=2, ff_dim=192, dropout=0.1, num_workers=0, amp=False, compile_model=False, seed=11))

## 1. Train the lightweight retriever on sampled interactions

In [3]:
corpus = lw.VideoCorpus(lw_cfg, seed=lw_cfg.seed)
train_dataset = lw.RetrievalDataset(lw_cfg.train_size, lw_cfg, corpus, seed=lw_cfg.seed + 1)
test_dataset = lw.RetrievalDataset(lw_cfg.test_size, lw_cfg, corpus, seed=lw_cfg.seed + 10_000)

train_loader = lw.build_loader(train_dataset, lw_cfg.batch_size, shuffle=True, cfg=lw_cfg, device=device)
test_loader = lw.build_loader(test_dataset, lw_cfg.batch_size, shuffle=False, cfg=lw_cfg, device=device)

lightweight_model = lw.build_model(lw_cfg, device)
lightweight_optimizer = torch.optim.AdamW(
    lightweight_model.parameters(),
    lr=lw_cfg.lr,
    weight_decay=lw_cfg.weight_decay,
)
lightweight_scaler = torch.cuda.amp.GradScaler(enabled=lw_cfg.amp and device.type == "cuda")

lightweight_history = []
for epoch in range(1, lw_cfg.epochs + 1):
    train_metrics = lw.run_epoch(
        lightweight_model,
        train_loader,
        device,
        lw_cfg,
        optimizer=lightweight_optimizer,
        scaler=lightweight_scaler,
    )
    eval_metrics = lw.evaluate_retrieval(lightweight_model, test_loader, corpus, lw_cfg, device)
    row = {"epoch": epoch, **train_metrics, **eval_metrics}
    lightweight_history.append(row)
    print(row)

lightweight_history[-1]

/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_88052/3039573797.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  lightweight_scaler = torch.cuda.amp.GradScaler(enabled=lw_cfg.amp and device.type == "cuda")


{'epoch': 1, 'loss': 4.49433571100235, 'batch_acc': 0.04531250015133992, 'recall@6': 0.21874999813735485, 'mrr': 0.14108767732977867}
{'epoch': 2, 'loss': 2.9999737441539764, 'batch_acc': 0.18828124925494194, 'recall@6': 0.453125, 'mrr': 0.2309742085635662}


{'epoch': 2,
 'loss': 2.9999737441539764,
 'batch_acc': 0.18828124925494194,
 'recall@6': 0.453125,
 'mrr': 0.2309742085635662}

## 2. Share the sampled catalog with the deep reranker

In [4]:
shared_catalog = dr.catalog_from_retrieval_corpus(corpus, dr_cfg)

sample_retrieval_dataset = lw.RetrievalDataset(8, lw_cfg, corpus, seed=lw_cfg.seed + 500)
sample_retrieval_loader = lw.build_loader(
    sample_retrieval_dataset,
    batch_size=8,
    shuffle=False,
    cfg=lw_cfg,
    device=device,
)
sample_retrieval_batch = next(iter(sample_retrieval_loader))
sample_topk = lw.retrieve_topk(lightweight_model, sample_retrieval_batch, corpus, lw_cfg, device)

reranker_batch = dr.build_reranker_batch_from_retrieval(
    sample_retrieval_batch,
    sample_topk["topk_video_ids"],
    shared_catalog,
    dr_cfg,
    seed=dr_cfg.seed,
)

{key: tuple(value.shape) for key, value in reranker_batch.items()}

{'user_id': (8,),
 'user_dense': (8, 10),
 'pref_topic': (8,),
 'pref_creator': (8,),
 'session_depth': (8,),
 'novelty_pref': (8,),
 'history_ids': (8, 37),
 'history_mask': (8, 37),
 'slate_ids': (8, 6),
 'slate_dense': (8, 6, 12),
 'slate_topic': (8, 6),
 'slate_creator': (8, 6),
 'slate_quality': (8, 6),
 'slate_freshness': (8, 6),
 'slate_length_bucket': (8, 6),
 'click_label': (8, 6),
 'watch_target': (8, 6)}

## 3. Generate end-to-end reranker training batches from retriever output

In [5]:
def make_reranker_batch(num_samples: int, seed: int) -> dict[str, torch.Tensor]:
    retrieval_dataset = lw.RetrievalDataset(num_samples, lw_cfg, corpus, seed=seed)
    retrieval_loader = lw.build_loader(
        retrieval_dataset,
        batch_size=min(num_samples, lw_cfg.batch_size),
        shuffle=False,
        cfg=lw_cfg,
        device=device,
    )
    retrieval_batch = next(iter(retrieval_loader))
    topk = lw.retrieve_topk(lightweight_model, retrieval_batch, corpus, lw_cfg, device)
    return dr.build_reranker_batch_from_retrieval(
        retrieval_batch,
        topk["topk_video_ids"],
        shared_catalog,
        dr_cfg,
        seed=seed,
    )


end_to_end_batch = make_reranker_batch(num_samples=12, seed=1234)
end_to_end_batch["slate_ids"][0], end_to_end_batch["click_label"][0]

(tensor([339, 134, 179,  26, 284,  54]), tensor([0., 0., 0., 0., 0., 1.]))

## 4. Train the deep reranker on top-k slates sampled from stage 1

In [6]:
deeprank_model = dr.build_model(dr_cfg, device)
deeprank_optimizer = torch.optim.AdamW(
    deeprank_model.parameters(),
    lr=dr_cfg.lr,
    weight_decay=dr_cfg.weight_decay,
)
deeprank_scaler = torch.cuda.amp.GradScaler(enabled=dr_cfg.amp and device.type == "cuda")


def train_deeprank_step(batch_seed: int) -> dict[str, float]:
    deeprank_model.train()
    batch = dr.move_batch_to_device(make_reranker_batch(dr_cfg.batch_size, batch_seed), device)
    deeprank_optimizer.zero_grad(set_to_none=True)
    outputs = deeprank_model(batch)
    loss, click_loss, watch_loss = dr.compute_loss(
        outputs,
        batch["click_label"],
        batch["watch_target"],
        alpha=dr_cfg.alpha,
        beta=dr_cfg.beta,
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(deeprank_model.parameters(), dr_cfg.grad_clip)
    deeprank_optimizer.step()
    hit_rate, mrr = dr.ranking_metrics(outputs, batch["click_label"])
    return {
        "loss": float(loss.item()),
        "click_loss": float(click_loss.item()),
        "watch_loss": float(watch_loss.item()),
        "hit_rate": hit_rate,
        "mrr": mrr,
    }


deeprank_history = []
for step in range(1, 16):
    metrics = train_deeprank_step(batch_seed=10_000 + step)
    deeprank_history.append({"step": step, **metrics})
    if step % 5 == 0:
        print(deeprank_history[-1])

deeprank_history[-1]

/Users/linghuang/Git/ML-System-Design/sampled-ml-code/video-recommendation/deepranker.py:299: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.history_encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg.num_layers)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_88052/3182729536.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  deeprank_scaler = torch.cuda.amp.GradScaler(enabled=dr_cfg.amp and device.type == "cuda")


{'step': 5, 'loss': 0.4787991940975189, 'click_loss': 0.47105953097343445, 'watch_loss': 0.030958639457821846, 'hit_rate': 0.2604166567325592, 'mrr': 0.49079862236976624}
{'step': 10, 'loss': 0.461109459400177, 'click_loss': 0.45424026250839233, 'watch_loss': 0.027476809918880463, 'hit_rate': 0.3645833432674408, 'mrr': 0.577256977558136}
{'step': 15, 'loss': 0.43512749671936035, 'click_loss': 0.4281390905380249, 'watch_loss': 0.02795363776385784, 'hit_rate': 0.4166666567325592, 'mrr': 0.6270833015441895}


{'step': 15,
 'loss': 0.43512749671936035,
 'click_loss': 0.4281390905380249,
 'watch_loss': 0.02795363776385784,
 'hit_rate': 0.4166666567325592,
 'mrr': 0.6270833015441895}

## 5. Compare stage-1 retrieval order with stage-2 reranking

In [7]:
eval_retrieval_dataset = lw.RetrievalDataset(5, lw_cfg, corpus, seed=77_777)
eval_retrieval_loader = lw.build_loader(
    eval_retrieval_dataset,
    batch_size=5,
    shuffle=False,
    cfg=lw_cfg,
    device=device,
)
eval_retrieval_batch = next(iter(eval_retrieval_loader))
eval_topk = lw.retrieve_topk(lightweight_model, eval_retrieval_batch, corpus, lw_cfg, device)
eval_reranker_batch = dr.build_reranker_batch_from_retrieval(
    eval_retrieval_batch,
    eval_topk["topk_video_ids"],
    shared_catalog,
    dr_cfg,
    seed=88_888,
)
reranked = dr.rerank_slate(deeprank_model, eval_reranker_batch, device)

for row in range(3):
    print(f"User {row}")
    print("  positive_video_id:", int(eval_retrieval_batch["positive_video_id"][row]))
    print("  retriever_topk:", eval_topk["topk_video_ids"][row].tolist())
    print("  reranked_topk:", reranked["ranked_video_ids"][row].tolist())
    print("  reranker_click_labels:", eval_reranker_batch["click_label"][row].tolist())
    print("  reranker_scores:", [round(float(x), 4) for x in reranked["scores"][row].tolist()])

User 0
  positive_video_id: 54
  retriever_topk: [180, 134, 339, 288, 186, 26]
  reranked_topk: [134, 54, 288, 180, 186, 339]
  reranker_click_labels: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
  reranker_scores: [0.114, 0.1657, 0.0962, 0.144, 0.1109, 0.1582]
User 1
  positive_video_id: 78
  retriever_topk: [256, 391, 289, 128, 181, 30]
  reranked_topk: [78, 391, 289, 128, 181, 256]
  reranker_click_labels: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
  reranker_scores: [0.1024, 0.1459, 0.1189, 0.1142, 0.1066, 0.1979]
User 2
  positive_video_id: 389
  retriever_topk: [71, 50, 389, 83, 33, 174]
  reranked_topk: [389, 33, 174, 83, 71, 50]
  reranker_click_labels: [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
  reranker_scores: [0.1011, 0.1003, 0.1157, 0.1012, 0.1118, 0.1115]
